# Interpreting and Simplifying Regular Expressions

This notebook implements two programs for the regular expressions that have been defined in Chapter 2 of the
lecture notes:
- An *interpreter*: In Section 2.2, the meaning of a regular expression $r$ is defined as a formal language
  $L(r)$.  The function `matches r s` checks whether the string `s` is an element of $L(r)$.
- A *simplifier*: In Section 2.3, we have seen a number of algebraic laws for regular expressions.  For
  example, the law
  $$ \varepsilon \cdot r \doteq r $$
  tells us that the regular expression $\varepsilon \cdot r$ describes the same language as the regular
  expression $r$.  If we read such a law from left to right, we can use it to *simplify* a regular expression.
  The function `simplify_regexp` simplifies regular expressions in this way.

The notebook consists of five parts:
1. A *scanner* splits a string like `'(ε + a)*·b'` into a list of tokens.
2. A hand-written *recursive descent parser* turns this list of tokens into an *abstract syntax tree*.

   For the moment, we will not discuss the *scanner* and the *parser*, as these are topics that will only be covered later in this lecture.
3. The function `to_str` turns an abstract syntax tree back into a string.
4. The function `matches` is an interpreter: it checks whether a string is an element of the language that is
   described by a regular expression.
5. The function `simplify_regexp` simplifies an abstract syntax tree using the laws of Section 2.3.

Finally, the last section shows how a type checker finds errors in the use of these functions.

**Contents**

1. [Type Checking](#Type-Checking)
2. [The Syntax of Regular Expressions](#The-Syntax-of-Regular-Expressions)
3. [The Scanner](#The-Scanner)
4. [Abstract Syntax Trees](#Abstract-Syntax-Trees)
5. [The Parser](#The-Parser)
6. [Turning Trees back into Strings](#Turning-Trees-back-into-Strings)
7. [An Interpreter for Regular Expressions](#An-Interpreter-for-Regular-Expressions)
8. [Sums and Products as Lists](#Sums-and-Products-as-Lists)
9. [Simplifying Products](#Simplifying-Products)
10. [Simplifying the Kleene Closure](#Simplifying-the-Kleene-Closure)
11. [Simplifying Sums](#Simplifying-Sums)
12. [Putting it all Together](#Putting-it-all-Together)
13. [Trying it Out](#Trying-it-Out)
14. [The Type Checker at Work](#The-Type-Checker-at-Work)

## Type Checking

All functions in this notebook carry *type annotations*.  In *Python*, these annotations are ignored when the
program runs, and a separate *static type checker* like *basedpyright* is needed to find type errors before the
program is run.  In *OCaml*, type checking is part of the language: every cell is type checked before it is
executed, and a cell that contains a type error is not executed at all.  The annotations are even
optional, since OCaml *infers* the type of every expression.  We write them anyway, since they document
the functions.  After a definition has been executed, the notebook shows the type that OCaml has computed, e.g.
```
val tokenize : string -> string list = <fun>
```
There is nothing to install: the OCaml kernel for *Jupyter* reports type errors with the line and the
column, and the error is marked in the cell.

The next cell loads the helper functions that are shared by all OCaml notebooks.  In this notebook, we only need
the function `repr`, which is explained below.

In [1]:
#use "../lib/nb.ml";;

- : unit = ()
Findlib has been successfully loaded. Additional directives:
  #require "package";;      to load a package
  #list;;                   to list the available packages
  #camlp4o;;                to load camlp4 (standard syntax)
  #camlp4r;;                to load camlp4 (revised syntax)
  #predicates "p,q,...";;   to set these predicates
  Topfind.reset();;         to force that packages will be reloaded
  #thread;;                 to enable threads

- : unit = ()
- : unit = ()
val lib_dir : string = "../lib"
val loaded_libs : (string, unit) Hashtbl.t = <abstr>
val load_lib : string -> unit = <fun>


## The Syntax of Regular Expressions

We use the syntax of Chapter 2:
- `∅` denotes the empty language and `ε` denotes the language that contains only the empty string.
- Every letter and every digit is a character of the alphabet $\Sigma$.
- `+` denotes the union of two languages, `·` denotes their product, and the postfix operator `*` denotes
  the Kleene closure.
- Parentheses can be used for grouping.

The postfix operator `*` has the highest precedence, followed by `·`, while `+` has the lowest precedence.
Both `+` and `·` associate to the left.  Blanks are ignored.

**Note:** In *Jupyter*, the symbols `∅` and `ε` can be entered by typing `\emptyset` and `\varepsilon`,
followed by the tabulator key.  Typing `\cdot` followed by the tabulator key yields the symbol `⋅`, which
looks almost like `·` but is a different Unicode character.  Therefore, our scanner accepts both of them.

## The Scanner

Every token of a regular expression consists of a single character.  Therefore, we do not need regular
expressions to implement our scanner: the function `tokenize s` looks at the characters of the string `s`
one by one.  OCaml strings are sequences of *bytes*, and the symbols `∅`, `ε`, `·`, and `⋅` need two or three
bytes in the encoding UTF-8.  Therefore, the function `String.get_utf_8_uchar s i` is used to *decode* the
character that starts at position `i`.  The result contains the character and the number of bytes it uses.
Every token is returned as a string that contains exactly one character.
- Blanks and other white space characters are skipped.
- The character `⋅` is replaced by `·`, so that the parser only has to deal with `·`.
- The function `is_alnum c` checks whether `c` is an ASCII letter or a digit.  Note that `ε` is not an
  ASCII letter, so there is no danger of confusing it with a character of our alphabet.  (Python's method
  `isalnum` also accepts Greek letters like `ε`.)
- Every other character is an error.

**Function `tokenize s`**
- *Input:* `s` is a string that represents a regular expression.
- *Output:* The list of the tokens of `s`.  Every token is a string that contains a single character.  The exception `Syntax_error` is raised if `s` contains an illegal character.

In Python, `SyntaxError` is a built-in exception.  In OCaml, we declare our own exception `Syntax_error`, which
carries the error message.

In [2]:
exception Syntax_error of string

let is_alnum (c : string) : bool =
  String.length c = 1 && match c.[0] with 'a' .. 'z' | 'A' .. 'Z' | '0' .. '9' -> true | _ -> false

let tokenize (s : string) : string list =
  let rec scan (i : int) : string list =
    if i >= String.length s then []
    else
      let d = String.get_utf_8_uchar s i in
      let n = Uchar.utf_decode_length d in
      let c = String.sub s i n in
      match c with
      | " " | "\t" | "\n" | "\r" -> scan (i + n)
      | "⋅" -> "·" :: scan (i + n)
      | "∅" | "ε" | "+" | "·" | "*" | "(" | ")" -> c :: scan (i + n)
      | _ when is_alnum c -> c :: scan (i + n)
      | _ -> raise (Syntax_error (Printf.sprintf "Illegal character %s in %s." (repr c) (repr s)))
  in
  scan 0

exception Syntax_error of string


val is_alnum : string -> bool = <fun>


val tokenize : string -> string list = <fun>


Let's test the scanner.

In [3]:
tokenize "(ε + a)* ⋅ b"

- : string list = ["("; "ε"; "+"; "a"; ")"; "*"; "·"; "b"]


## Abstract Syntax Trees

In *Python*, regular expressions are represented as *nested tuples* that are built from strings and tuples:
`∅` and `ε` are represented by the strings `'∅'` and `'ε'`, a character `c` by the string `c`, and
$r_1 + r_2$ by the tuple `('+', t1, t2)`.

In *OCaml*, we define a new type `ast` instead.  Its values are known as *abstract syntax trees*:
- The regular expressions `∅` and `ε` are represented by the *constructors* `Empty` and `Epsilon`.
- A character `c` from the alphabet is represented by `Char c`.
- The regular expression $r_1 + r_2$ is represented by `Sum (t1, t2)`, where `t1` and `t2` are
  the representations of $r_1$ and $r_2$.
- The regular expression $r_1 \cdot r_2$ is represented by `Product (t1, t2)`.
- The regular expression $r^*$ is represented by `Star t`, where `t` is the representation of $r$.

Parentheses do not show up in the abstract syntax tree, since $L\bigl((r)\bigr) = L(r)$.  They are only
needed to determine the structure of the tree.  For example, `a + b·c*` is represented by
```
Sum (Char 'a', Product (Char 'b', Star (Char 'c')))
```

The type `ast` is a *variant type*: every value of type `ast` is built with exactly one of the six constructors.

**Type safety:** The Python type `str | tuple[...]` admits the string `'ε'` both as the regular expression
$\varepsilon$ and, since `ε` is a letter, as a character.  Strings like `'ab'` are admitted, too.  In OCaml,
`Epsilon` and `Char c` are different constructors, and `Char` takes a value of type `char`, which is exactly
one character.  Moreover, when a function uses `match` on an `ast`, the compiler warns if one of the
constructors is not handled.

In [4]:
type ast =
  | Empty                   (* ∅      *)
  | Epsilon                 (* ε      *)
  | Char    of char         (* c      *)
  | Star    of ast          (* r*     *)
  | Sum     of ast * ast    (* r1 + r2 *)
  | Product of ast * ast    (* r1 · r2 *)

type ast =
    Empty
  | Epsilon
  | Char of char
  | Star of ast
  | Sum of ast * ast
  | Product of ast * ast


## The Parser

Our parser is a *recursive descent parser* that is based on the following grammar:
$$
  \begin{array}{lcl}
  \mathrm{regexp}  & \rightarrow & \mathrm{product}\;\;\bigl(\texttt{'+'}\;\; \mathrm{product}\bigr)^* \\[0.2cm]
  \mathrm{product} & \rightarrow & \mathrm{factor} \;\;\bigl(\texttt{'·'}\;\; \mathrm{factor}\bigr)^*  \\[0.2cm]
  \mathrm{factor}  & \rightarrow & \mathrm{atom} \;\;\texttt{'*'}^*                                    \\[0.2cm]
  \mathrm{atom}    & \rightarrow & \texttt{'('} \;\;\mathrm{regexp} \;\;\texttt{')'}
                     \;\mid\; \texttt{'∅'} \;\mid\; \texttt{'ε'} \;\mid\; \texttt{CHAR}
  \end{array}
$$
Here, the operator ${}^*$ in the grammar means that the part before it can be repeated any number of times.
For every grammar symbol there is a function that parses this grammar symbol.  Every one of these functions
takes a list of tokens `tl` as its argument and returns a pair `(tree, rest)`, where
- `tree` is the representation of the part of `tl` that has been parsed, and
- `rest` is the list of those tokens that have not yet been consumed.

These functions are *mutually recursive*: `parse_regexp` calls `parse_product`, which is defined later, and
`parse_atom` calls `parse_regexp` again.  In OCaml, a function can only use functions that have already been
defined, unless all of them are defined together using `let rec ... and ...`.  To keep every function in a cell
of its own, we use *forward references*: the next cell creates references (i.e. mutable boxes) that will
later hold the functions `parse_product`, `parse_factor`, and `parse_atom`.  `!parse_product_ref` is the
function that is currently stored in the box.  When a function has been defined, it is stored in its box with
the operator `:=`.

The function `parse_regexp tl` implements the rule for $\mathrm{regexp}$:
- First, it parses a product.
- As long as the next token is `+`, it parses another product and combines the tree built so far with the
  new tree.  Since the new tree becomes the right argument of `+`, the operator `+` associates to the left.
  The `while` loop of Python is replaced by the recursive function `loop`.

In [5]:
let not_yet_defined _ = failwith "not yet defined"

let parse_product_ref : (string list -> ast * string list) ref = ref not_yet_defined
let parse_factor_ref  : (string list -> ast * string list) ref = ref not_yet_defined
let parse_atom_ref    : (string list -> ast * string list) ref = ref not_yet_defined

val not_yet_defined : 'a -> 'b = <fun>


val parse_product_ref : (string list -> ast * string list) ref =
  {contents = <fun>}


val parse_factor_ref : (string list -> ast * string list) ref =
  {contents = <fun>}


val parse_atom_ref : (string list -> ast * string list) ref =
  {contents = <fun>}


**Function `parse_regexp tl`**
- *Input:* `tl` is a list of tokens.
- *Output:* A pair `(tree, rest)`, where `tree` is the abstract syntax tree of the regular expression at the beginning of `tl` and `rest` is the list of the remaining tokens.

In [6]:
let parse_regexp (tl : string list) : ast * string list =
  let rec loop result rest =
    match rest with
    | "+" :: rest ->
        let arg, rest = !parse_product_ref rest in
        loop (Sum (result, arg)) rest
    | _ -> result, rest
  in
  let result, rest = !parse_product_ref tl in
  loop result rest

val parse_regexp : string list -> ast * string list = <fun>


The function `parse_product tl` implements the rule for $\mathrm{product}$ in the same way.

**Function `parse_product tl`**
- *Input:* `tl` is a list of tokens.
- *Output:* A pair `(tree, rest)`, where `tree` is the abstract syntax tree of the product at the beginning of `tl` and `rest` is the list of the remaining tokens.

In [7]:
let parse_product (tl : string list) : ast * string list =
  let rec loop result rest =
    match rest with
    | "·" :: rest ->
        let arg, rest = !parse_factor_ref rest in
        loop (Product (result, arg)) rest
    | _ -> result, rest
  in
  let result, rest = !parse_factor_ref tl in
  loop result rest
;;
parse_product_ref := parse_product

val parse_product : string list -> ast * string list = <fun>


- : unit = ()


The function `parse_factor tl` implements the rule for $\mathrm{factor}$.  It first parses an atom.  Then,
every `*` that follows wraps the tree built so far into `Star tree`.  Hence, `a**` is
represented as `Star (Star (Char 'a'))`.

**Function `parse_factor tl`**
- *Input:* `tl` is a list of tokens.
- *Output:* A pair `(tree, rest)`, where `tree` is the abstract syntax tree of the factor at the beginning of `tl` and `rest` is the list of the remaining tokens.

In [8]:
let parse_factor (tl : string list) : ast * string list =
  let rec loop result rest =
    match rest with
    | "*" :: rest -> loop (Star result) rest
    | _ -> result, rest
  in
  let result, rest = !parse_atom_ref tl in
  loop result rest
;;
parse_factor_ref := parse_factor

val parse_factor : string list -> ast * string list = <fun>


- : unit = ()


The function `parse_atom tl` implements the rule for $\mathrm{atom}$.
- If the first token is an opening parenthesis, we parse a regular expression, which has to be followed by a
  closing parenthesis.
- The tokens `∅` and `ε` are turned into `Empty` and `Epsilon`, and a character `c` is turned into `Char c`.
  The pattern `c :: rest when is_alnum c` matches a list whose first element `c` is a letter or a digit.
- Every other token, e.g. `+` or `)`, cannot start an atom and hence is an error.  If the list `tl` is
  empty, the input has ended too early.

Instead of looking at `tl[0]` and `tl[1:]`, the OCaml function uses *pattern matching* on lists: the pattern
`"(" :: rest` matches every list whose first element is `"("` and binds `rest` to the remaining elements.

**Function `parse_atom tl`**
- *Input:* `tl` is a list of tokens.
- *Output:* A pair `(tree, rest)`, where `tree` is the abstract syntax tree of the atom at the beginning of `tl` and `rest` is the list of the remaining tokens.  The exception `Syntax_error` is raised if `tl` does not start with an atom.

In [9]:
let parse_atom (tl : string list) : ast * string list =
  match tl with
  | [] -> raise (Syntax_error "Unexpected end of input.")
  | "(" :: rest -> (
      let result, rest = parse_regexp rest in
      match rest with
      | ")" :: rest -> result, rest
      | [] -> raise (Syntax_error "')' expected, but found [].")
      | t :: _ -> raise (Syntax_error (Printf.sprintf "')' expected, but found [%s]." (repr t))))
  | "∅" :: rest -> Empty, rest
  | "ε" :: rest -> Epsilon, rest
  | c :: rest when is_alnum c -> Char c.[0], rest
  | t :: _ -> raise (Syntax_error (Printf.sprintf "Unexpected token %s." (repr t)))
;;
parse_atom_ref := parse_atom

val parse_atom : string list -> ast * string list = <fun>


- : unit = ()


The function `parse s` puts the pieces together: it tokenizes the string `s`, parses the resulting list of
tokens, and checks that all tokens have been consumed.

**Function `parse s`**
- *Input:* `s` is a string that represents a regular expression.
- *Output:* The abstract syntax tree of `s`.  The exception `Syntax_error` is raised if `s` is not a regular expression.

In [10]:
let parse (s : string) : ast =
  match parse_regexp (tokenize s) with
  | tree, [] -> tree
  | _, t :: _ -> raise (Syntax_error (Printf.sprintf "Unexpected token %s." (repr t)))

val parse : string -> ast = <fun>


In [11]:
parse "a + b·c*"

- : ast = Sum (Char 'a', Product (Char 'b', Star (Char 'c')))


Let us check that `+` and `·` associate to the left.

In [12]:
parse "a + b + c"

- : ast = Sum (Sum (Char 'a', Char 'b'), Char 'c')


In [13]:
parse "a·b·c"

- : ast = Product (Product (Char 'a', Char 'b'), Char 'c')


## Turning Trees back into Strings

The function `to_str r` turns the abstract syntax tree `r` into a string.  It only uses parentheses where
they are needed.  To decide this, the function `precedence r` returns the precedence of the operator at the
root of `r`: atoms bind tighter than every operator, so they get the highest precedence.

**Function `precedence r`**
- *Input:* `r` is an abstract syntax tree.
- *Output:* The precedence of the operator at the root of `r`.  Atoms have the precedence $4$.

In [14]:
let precedence (r : ast) : int =
  match r with
  | Empty | Epsilon | Char _ -> 4
  | Star _                   -> 3
  | Product _                -> 2
  | Sum _                    -> 1

val precedence : ast -> int = <fun>


The function `to_str` uses the helper function `arg_str r p`, which puts parentheses around the string of
`r` if the precedence of `r` is lower than `p`.
- The argument of `*` needs parentheses unless it is an atom or another `*` expression.
- The left argument of a binary operator needs parentheses if its precedence is lower than the precedence of
  the operator.
- Since `+` and `·` associate to the left, the right argument needs parentheses already if its precedence is
  the same as the precedence of the operator.  For example, the tree `Sum (Char 'a', Sum (Char 'b', Char 'c'))` is printed
  as `a + (b + c)`.

The functions `arg_str` and `to_str` call each other.  This time, we define them together in one cell using
`let rec ... and ...`.

**Function `arg_str r p`**
- *Input:* `r` is an abstract syntax tree and `p` is a precedence.
- *Output:* The string representing `r`, enclosed in parentheses if the precedence of `r` is lower than `p`.

**Function `to_str r`**
- *Input:* `r` is an abstract syntax tree.
- *Output:* The string representing `r`, using as few parentheses as possible.

In [15]:
let rec arg_str (r : ast) (p : int) : string =
  if precedence r < p then "(" ^ to_str r ^ ")"
  else to_str r

and to_str (r : ast) : string =
  match r with
  | Empty -> "∅"
  | Epsilon -> "ε"
  | Char c -> String.make 1 c
  | Star r1 -> arg_str r1 3 ^ "*"
  | Sum (r1, r2) | Product (r1, r2) ->
      let p   = precedence r in
      let sep = match r with Sum _ -> " + " | _ -> "·" in
      arg_str r1 p ^ sep ^ arg_str r2 (p + 1)

val arg_str : ast -> int -> string = <fun>
val to_str : ast -> string = <fun>


In [16]:
to_str (parse "(a + b + c)·c*·(d·(e·f))*")

- : string = "(a + b + c)·c*·(d·(e·f))*"


In [17]:
to_str (Sum (Char 'a', Sum (Char 'b', Char 'c')))

- : string = "a + (b + c)"


## An Interpreter for Regular Expressions

In Chapter 2, the meaning of a regular expression $r$ is defined as the language $L(r)$.  The function
`matches r s` checks whether the string `s` is an element of the language $L(r)$ of the regular expression
`r`, which is given as a string.  It parses `r` and then calls the function `member s t`, which does the
real work on the abstract syntax tree `t`.

The function `member s t` follows the inductive definition of $L(t)$ in Section 2.2 of the lecture notes.
It uses a `match` expression: every case compares the tree `t` with a *pattern*.  A pattern like
`Sum (r1, r2)` matches every tree built with the constructor `Sum` and binds the variables `r1` and `r2` to
its arguments.  Since every constructor has its own pattern, the order of the cases does not matter here.
The cases correspond to the clauses of the definition:
- $L(\emptyset) = \{\}$: no string is an element of the empty language.
- $L(\varepsilon) = \{\lambda\}$: only the empty string is an element.
- $L(c) = \{c\}$: only the string consisting of the character $c$ is an element.
- $L(r_1 + r_2) = L(r_1) \cup L(r_2)$: the string `s` has to be an element of $L(r_1)$ or of $L(r_2)$.
- $L(r_1 \cdot r_2) = L(r_1) \cdot L(r_2)$: the string `s` has to be split into two parts `prefix s k` and
  `suffix s k`, i.e. the first `k` characters and the rest, such that `prefix s k` is an element of $L(r_1)$
  and `suffix s k` is an element of $L(r_2)$.  We try every position `k` from $0$ up to `String.length s`.
- $L(r^*) = \bigcup_{n \in \mathbb{N}} L(r)^n$: the string `s` is an element if it is empty, since
  $\lambda \in L(r)^0$.  Otherwise, `s` has to be split into a first part `prefix s k` that is an element of
  $L(r)$ and a rest `suffix s k` that is an element of $L(r^*)$.  Here, we only try positions `k` from $1$ on:
  if the first part were empty, the rest would be the string `s` itself, and the recursive call would never
  terminate.  We do not lose anything this way, since empty parts do not contribute to the string `s`.

Trying all positions `k` can lead to the same question being asked many times.  For example, when matching
`(a + a·a)*` against a long string of `a`s, the same suffix of `s` is checked over and over again.  Python
offers the *decorator* `@cache` from the module `functools` for this problem.  OCaml has no decorators, but
we can write the function `cache` ourselves: `cache f` returns a function that stores the result of every
call in a *hash table* and returns the stored result when it is called again with the same argument.
Since `member` is recursive, the recursive calls have to go through the cache, too.  Therefore, `f` gets the
cached function as its first argument `self`.  This works since abstract syntax trees and strings can be
used as keys of a hash table.

The next cell also defines the helper functions `range`, `prefix`, and `suffix`: `range a b` is the list
$[a, a+1, \cdots, b-1]$, just like Python's `range(a, b)`.

In [18]:
let cache (f : ('a -> 'b) -> 'a -> 'b) : 'a -> 'b =
  let table = Hashtbl.create 1024 in
  let rec self x =
    match Hashtbl.find_opt table x with
    | Some y -> y
    | None ->
        let y = f self x in
        Hashtbl.add table x y;
        y
  in
  self

let range (a : int) (b : int) : int list = List.init (max 0 (b - a)) (fun i -> a + i)
let prefix (s : string) (k : int) : string = String.sub s 0 k
let suffix (s : string) (k : int) : string = String.sub s k (String.length s - k)

val cache : (('a -> 'b) -> 'a -> 'b) -> 'a -> 'b = <fun>


val range : int -> int -> int list = <fun>


val prefix : string -> int -> string = <fun>


val suffix : string -> int -> string = <fun>


**Function `member s t`**
- *Input:* `s` is a string and `t` is the abstract syntax tree of a regular expression.
- *Output:* `true` if `s` is an element of the language $L(t)$, `false` otherwise.

Since `cache` works with functions of one argument, the cached function takes the pair `(s, t)`.  The
function `member` then turns it back into a function with two arguments.

In [19]:
let member_cached : string * ast -> bool =
  cache (fun member (s, t) ->
    match t with
    | Empty            -> false
    | Epsilon          -> s = ""
    | Char c           -> s = String.make 1 c
    | Sum (r1, r2)     -> member (s, r1) || member (s, r2)
    | Product (r1, r2) ->
        List.exists (fun k -> member (prefix s k, r1) && member (suffix s k, r2))
                    (range 0 (String.length s + 1))
    | Star r           ->
        s = "" || List.exists (fun k -> member (prefix s k, r) && member (suffix s k, t))
                              (range 1 (String.length s + 1)))

let member (s : string) (t : ast) : bool = member_cached (s, t)

val member_cached : string * ast -> bool = <fun>


val member : string -> ast -> bool = <fun>


**Function `matches r s`**
- *Input:* `r` is a string that represents a regular expression and `s` is a string.
- *Output:* `true` if `s` is an element of the language described by `r`, `false` otherwise.

In [20]:
let matches (r : string) (s : string) : bool =
  member s (parse r)

val matches : string -> string -> bool = <fun>


### Testing the Interpreter

The list `tests` contains triples of the form `(r, s, expected)`, where `expected` is the result that
`matches r s` should return.  Most of the regular expressions are taken from Chapters 1 and 2 of the
lecture notes:
- $(\mathtt{a} + \mathtt{b} + \mathtt{c}) \cdot (\mathtt{a} + \mathtt{b} + \mathtt{c})$ describes all
  strings of length $2$.
- $(\mathtt{a} + \mathtt{b} + \mathtt{c}) \cdot (\mathtt{a} + \mathtt{b} + \mathtt{c})^*$ describes all
  strings of length at least $1$.
- $(\mathtt{b} + \mathtt{c})^* \cdot \mathtt{a} \cdot (\mathtt{b} + \mathtt{c})^*$ describes all strings that
  contain exactly one `a`.
- $\mathtt{1} \cdot (\mathtt{0} + \mathtt{1})^* + \mathtt{0}$ describes the language $L_\mathbb{N}$ of
  Chapter 1, i.e. the natural numbers in binary notation.

In [21]:
let tests : (string * string * bool) list =
  [ ("∅",                                    "",        false);
    ("ε",                                    "",        true );
    ("ε",                                    "a",       false);
    ("a",                                    "a",       true );
    ("a",                                    "aa",      false);
    ("∅*",                                   "",        true );
    ("(a + b + c)·(a + b + c)",              "ab",      true );
    ("(a + b + c)·(a + b + c)",              "abc",     false);
    ("(a + b + c)·(a + b + c)*",             "",        false);
    ("(a + b + c)·(a + b + c)*",             "cab",     true );
    ("(b + c)*·a·(b + c)*",                  "bcacb",   true );
    ("(b + c)*·a·(b + c)*",                  "bcb",     false);
    ("(b + c)*·a·(b + c)*",                  "abca",    false);
    ("1·(0 + 1)* + 0",                       "100",     true );
    ("1·(0 + 1)* + 0",                       "010",     false);
    ("1·(0 + 1)* + 0",                       "0",       true );
    ("(a·b)*",                               "ababab",  true );
    ("(a·b)*",                               "aba",     false);
    ("(ε + a)*",                             "aaa",     true );
    ("(a + a·a)*·b",                         String.make 50 'a' ^ "b", true );
    ("(a + a·a)*·b",                         String.make 50 'a',  false);
  ]

val tests : (string * string * bool) list =
  [("∅", "", false); ("ε", "", true); ("ε", "a", false); ("a", "a", true);
   ("a", "aa", false); ("∅*", "", true);
   ("(a + b + c)·(a + b + c)", "ab", true);
   ("(a + b + c)·(a + b + c)", "abc", false);
   ("(a + b + c)·(a + b + c)*", "", false);
   ("(a + b + c)·(a + b + c)*", "cab", true);
   ("(b + c)*·a·(b + c)*", "bcacb", true);
   ("(b + c)*·a·(b + c)*", "bcb", false);
   ("(b + c)*·a·(b + c)*", "abca", false); ("1·(0 + 1)* + 0", "100", true);
   ("1·(0 + 1)* + 0", "010", false); ("1·(0 + 1)* + 0", "0", true);
   ("(a·b)*", "ababab", true); ("(a·b)*", "aba", false);
   ("(ε + a)*", "aaa", true);
   ("(a + a·a)*·b", "aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaab",
    true);
   ("(a + a·a)*·b", "aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa",
    false)]


The function `run_tests` checks every triple and prints the result.  If a result differs from the expected
one, the line is marked with `ERROR`.  For long strings, only the first $20$ characters are shown.

The function `repr`, which is defined in `../lib/nb.ml`, writes a string in OCaml syntax, i.e. enclosed in
double quotes, just like Python's `repr` does with single quotes.  The
formats `%-25s` and `%-26s` would pad a string with blanks, but they count *bytes*, not characters.  Since
`·` and `∅` need two or three bytes, the helper function `ljust s n`, which corresponds to Python's
`f'{s:n}'`, counts the characters instead.  The bytes of the form `0b10xxxxxx` are continuation
bytes of UTF-8, so they are not counted.

**Function `run_tests tests`**
- *Input:* `tests` is a list of triples `(r, s, expected)`, where `r` is a regular expression, `s` is a string, and `expected` is the result that `matches r s` should return.
- *Output:* `()`.  The function prints one line for every test and marks wrong results with `ERROR`.

In [22]:
let utf8_length (s : string) : int =
  String.fold_left (fun n c -> if Char.code c land 0xC0 = 0x80 then n else n + 1) 0 s

let ljust (s : string) (n : int) : string =
  s ^ String.make (max 0 (n - utf8_length s)) ' '

let run_tests (tests : (string * string * bool) list) : unit =
  List.iter
    (fun (r, s, expected) ->
      let result = matches r s in
      let mark   = if result = expected then "" else "   ERROR" in
      let shown  = if String.length s <= 20 then s else prefix s 20 ^ "..." in
      Printf.printf "%s %s %b%s\n%!" (ljust r 25) (ljust (repr shown) 26) result mark)
    tests

val utf8_length : string -> int = <fun>


val ljust : string -> int -> string = <fun>


val run_tests : (string * string * bool) list -> unit = <fun>


In [23]:
run_tests tests

∅                         ""                         false
ε                         ""                         true
ε                         "a"                        false
a                         "a"                        true
a                         "aa"                       false
∅*                        ""                         true
(a + b + c)·(a + b + c)   "ab"                       true
(a + b + c)·(a + b + c)   "abc"                      false
(a + b + c)·(a + b + c)*  ""                         false
(a + b + c)·(a + b + c)*  "cab"                      true
(b + c)*·a·(b + c)*       "bcacb"                    true
(b + c)*·a·(b + c)*       "bcb"                      false
(b + c)*·a·(b + c)*       "abca"                     false
1·(0 + 1)* + 0            "100"                      true
1·(0 + 1)* + 0            "010"                      false
1·(0 + 1)* + 0            "0"                        true
(a·b)*                    "ababab"                   true
(a·b)*

- : unit = ()


## Sums and Products as Lists

Because of the laws

(b) $\;(r_1 + r_2) + r_3 \doteq r_1 + (r_2 + r_3)$ and

(c) $\;(r_1 \cdot r_2) \cdot r_3 \doteq r_1 \cdot (r_2 \cdot r_3)$,

it does not matter how the parentheses are set in a sum or a product of several regular expressions.
Therefore, it is convenient to view a sum as a list of *alternatives* and a product as a list of *factors*.
- The function `summands r` returns the list of all regular expressions that are combined with `+` at the top
  of `r`.  If the top operator of `r` is not `+`, the list only contains `r` itself.
- The function `factors r` does the same for `·`.
- The function `make_sum l` builds a sum from a list of regular expressions.  The sum of an empty list is
  `∅`, since $L(\emptyset) = \{\}$ is the neutral element of the union.
- The function `make_product l` builds a product.  The product of an empty list is `ε`, since
  $L(\varepsilon) = \{\lambda\}$ is the neutral element of the product of languages.

Both `make_sum` and `make_product` build trees that associate to the left, just like the parser does.

**Function `summands r`**
- *Input:* `r` is an abstract syntax tree.
- *Output:* The list of the regular expressions that are combined with `+` at the top of `r`.

In [24]:
let rec summands (r : ast) : ast list =
  match r with
  | Sum (r1, r2) -> summands r1 @ summands r2
  | _ -> [ r ]

val summands : ast -> ast list = <fun>


**Function `factors r`**
- *Input:* `r` is an abstract syntax tree.
- *Output:* The list of the regular expressions that are combined with `·` at the top of `r`.

In [25]:
let rec factors (r : ast) : ast list =
  match r with
  | Product (r1, r2) -> factors r1 @ factors r2
  | _ -> [ r ]

val factors : ast -> ast list = <fun>


**Function `make_sum l`**
- *Input:* `l` is a list of abstract syntax trees.
- *Output:* The sum of the elements of `l`, associating to the left.  The sum of the empty list is `Empty`.

In [26]:
let make_sum (l : ast list) : ast =
  match l with
  | [] -> Empty
  | r :: rs -> List.fold_left (fun result r -> Sum (result, r)) r rs

val make_sum : ast list -> ast = <fun>


**Function `make_product l`**
- *Input:* `l` is a list of abstract syntax trees.
- *Output:* The product of the elements of `l`, associating to the left.  The product of the empty list is `Epsilon`.

In [27]:
let make_product (l : ast list) : ast =
  match l with
  | [] -> Epsilon
  | r :: rs -> List.fold_left (fun result r -> Product (result, r)) r rs

val make_product : ast list -> ast = <fun>


In [28]:
summands (parse "a + (b + c)·d + e")

- : ast list =
[Char 'a'; Product (Sum (Char 'b', Char 'c'), Char 'd'); Char 'e']


In [29]:
factors (parse "a·(b·c)·d*")

- : ast list = [Char 'a'; Char 'b'; Char 'c'; Star (Char 'd')]


Finally, the function `size r` counts the nodes of the tree `r`.  We use it to decide whether a
transformation really makes a regular expression simpler.

**Function `size r`**
- *Input:* `r` is an abstract syntax tree.
- *Output:* The number of nodes of `r`.

In [30]:
let rec size (r : ast) : int =
  match r with
  | Empty | Epsilon | Char _ -> 1
  | Star r1 -> 1 + size r1
  | Sum (r1, r2) | Product (r1, r2) -> 1 + size r1 + size r2

val size : ast -> int = <fun>


## Simplifying Products

The function `simplify_product fs` simplifies a product, which is given as the list `fs` of its factors.
These factors have already been simplified.
- By law (d), $\emptyset \cdot r \doteq r \cdot \emptyset \doteq \emptyset$, a product that contains `∅`
  as a factor is equivalent to `∅`.
- By law (e), $\varepsilon \cdot r \doteq r \cdot \varepsilon \doteq r$, all factors that are equal to `ε`
  can be dropped.  If no factor remains, `make_product` returns `ε`.

**Function `simplify_product fs`**
- *Input:* `fs` is the list of the factors of a product.  These factors are already simplified.
- *Output:* A simplified abstract syntax tree that is equivalent to the product of the factors in `fs`.

In [31]:
let simplify_product (fs : ast list) : ast =
  if List.mem Empty fs then Empty
  else make_product (List.filter (fun f -> f <> Epsilon) fs)

val simplify_product : ast list -> ast = <fun>


## Simplifying the Kleene Closure

The function `simplify_star s` simplifies the regular expression $s^*$.  The argument `s` has already been
simplified.
- By the laws (k), $\emptyset^* \doteq \varepsilon$, and (l), $\varepsilon^* \doteq \varepsilon$, the
  closure of `∅` and of `ε` is `ε`.
- By law (j), $(r^*)^* \doteq r^*$, the closure of a closure is the closure itself.
- By law (n), $(\varepsilon + r)^* \doteq r^*$, the summand `ε` can be dropped from a sum that occurs as the
  argument of `*`.  Since the sum might then consist of a single summand, which might be a closure itself,
  we call `simplify_star` again.

**Function `simplify_star s`**
- *Input:* `s` is an abstract syntax tree that is already simplified.
- *Output:* A simplified abstract syntax tree that is equivalent to $s^*$.

In [32]:
let rec simplify_star (s : ast) : ast =
  match s with
  | Empty | Epsilon -> Epsilon
  | Star _ -> s
  | _ ->
      let alternatives = summands s in
      if List.mem Epsilon alternatives then
        simplify_star (make_sum (List.filter (fun a -> a <> Epsilon) alternatives))
      else Star s

val simplify_star : ast -> ast = <fun>


## Simplifying Sums

Sums need the most work.  The function `simplify_sum alts` simplifies a sum, which is given as the list
`alts` of its summands.  These summands have already been simplified.
- By law (f), $\emptyset + r \doteq r + \emptyset \doteq r$, all summands that are equal to `∅` are dropped.
- By law (i), $r + r \doteq r$, a summand that occurs twice is dropped.  Because of law (a),
  $r_1 + r_2 \doteq r_2 + r_1$, this works even when the two occurrences are not next to each other.
- If `ε` is one of the summands, the function `absorb_epsilon` tries to get rid of it.  This is explained
  below.
- Finally, the function `factor_out` uses the laws of distributivity to factor out common parts of the
  summands.  This is explained below, too.

The functions `absorb_epsilon` and `factor_out` are only defined below, and `factor_out` calls `simplify`,
which is defined at the very end.  As in the parser, we use forward references for these three functions.

In [33]:
let absorb_epsilon_ref : (ast list -> ast list) ref = ref not_yet_defined
let factor_out_ref     : (ast list -> ast) ref      = ref not_yet_defined
let simplify_ref       : (ast -> ast) ref           = ref not_yet_defined

val absorb_epsilon_ref : (ast list -> ast list) ref = {contents = <fun>}


val factor_out_ref : (ast list -> ast) ref = {contents = <fun>}


val simplify_ref : (ast -> ast) ref = {contents = <fun>}


**Function `simplify_sum alts`**
- *Input:* `alts` is the list of the summands of a sum.  These summands are already simplified.
- *Output:* A simplified abstract syntax tree that is equivalent to the sum of the summands in `alts`.

In [34]:
let simplify_sum (alts : ast list) : ast =
  let unique =
    List.fold_left (fun unique a -> if a <> Empty && not (List.mem a unique) then unique @ [ a ] else unique)
                   [] alts
  in
  let unique = if List.mem Epsilon unique then !absorb_epsilon_ref unique else unique in
  !factor_out_ref unique

val simplify_sum : ast list -> ast = <fun>


### Absorbing `ε`

Law (m) states that $\varepsilon + r^* \cdot r \doteq r^*$, i.e. the sum $\varepsilon + r^* \cdot r$ can be
replaced by $r^*$.  Because of the laws (a) and (b), the summands
$\varepsilon$ and $r^* \cdot r$ do not have to be next to each other.

In addition, the summand $\varepsilon$ can be dropped if another summand is a closure $r^*$, since
$\varepsilon \in L(r^*)$.  This is not one of the laws of Section 2.3, but it follows from them:
$$
  \varepsilon + r^* \;\stackrel{(m)}{\doteq}\; \varepsilon + (\varepsilon + r^* \cdot r)
               \;\stackrel{(b)}{\doteq}\; (\varepsilon + \varepsilon) + r^* \cdot r
               \;\stackrel{(i)}{\doteq}\; \varepsilon + r^* \cdot r
               \;\stackrel{(m)}{\doteq}\; r^*.
$$

The function `absorb_epsilon alts` gets a list of summands that contains `ε`.
- The function `is_star_times_base a` checks whether `a` has the form $r^* \cdot r$.  If `a` is a product
  with the factors $f_1, \cdots, f_n$, this is the case if $f_1$ has the form $r^*$ and the factors of $r$
  are $f_2, \cdots, f_n$.  For example, `(a·b)*·a·b` has this form.
- If one of the summands has the form $r^* \cdot r$, it is replaced by $r^*$ and `ε` is dropped.
- If one of the summands is a closure, `ε` is dropped.

**Function `is_star_times_base a`**
- *Input:* `a` is an abstract syntax tree.
- *Output:* `true` if `a` has the form $r^* \cdot r$, `false` otherwise.

In [35]:
let is_star_times_base (a : ast) : bool =
  match factors a with
  | Star r :: rest -> rest <> [] && factors r = rest
  | _ -> false

val is_star_times_base : ast -> bool = <fun>


**Function `absorb_epsilon alts`**
- *Input:* `alts` is a list of summands that contains `Epsilon`.
- *Output:* An equivalent list of summands, where `Epsilon` has been removed if law (m) or a closure allows this.

In [36]:
let absorb_epsilon (alts : ast list) : ast list =
  let drop_epsilon = List.filter (fun b -> b <> Epsilon) in
  match List.find_index is_star_times_base alts with
  | Some i ->
      drop_epsilon (List.mapi (fun j b -> if j = i then List.hd (factors b) else b) alts)
  | None ->
      if List.exists (function Star _ -> true | _ -> false) alts then drop_epsilon alts
      else alts
;;
absorb_epsilon_ref := absorb_epsilon

val absorb_epsilon : ast list -> ast list = <fun>


- : unit = ()


### Factoring out Common Parts

The laws (g) and (h) are the laws of distributivity:

(g) $\;(r_1 + r_2) \cdot r_3 \doteq r_1 \cdot r_3 + r_2 \cdot r_3$,

(h) $\;r_1 \cdot (r_2 + r_3) \doteq r_1 \cdot r_2 + r_1 \cdot r_3$.

Read from right to left, they allow us to factor out a common *suffix* or a common *prefix* of two summands.
For example, $\mathtt{a} \cdot \mathtt{b} + \mathtt{a} \cdot \mathtt{c}$ can be simplified to
$\mathtt{a} \cdot (\mathtt{b} + \mathtt{c})$.

The function `common_prefix fs1 fs2` returns the length of the longest common prefix of the lists `fs1` and
`fs2`.

**Function `common_prefix fs1 fs2`**
- *Input:* `fs1` and `fs2` are lists of abstract syntax trees.
- *Output:* The length of the longest common prefix of `fs1` and `fs2`.

In [37]:
let rec common_prefix (fs1 : ast list) (fs2 : ast list) : int =
  match fs1, fs2 with
  | f1 :: rest1, f2 :: rest2 when f1 = f2 -> 1 + common_prefix rest1 rest2
  | _ -> 0

val common_prefix : ast list -> ast list -> int = <fun>


The function `factor_candidates alts` generates all sums that result from factoring out a common prefix or
a common suffix of two summands.
- For every pair of summands `alts`$_i$ and `alts`$_j$, we compute the lists `fi` and `fj` of their factors.
- If these lists have a common prefix of length `k > 0`, we build the product of this prefix with the sum of
  the remaining factors.  For example, for the summands `a·b` and `a·c`, we build `a·(b + c)`.  If all
  factors of a summand belong to the prefix, the remaining factors form the product `ε`: `a·b + a` becomes
  `a·(b + ε)`.
- To handle a common suffix, we reverse the lists of factors, compute the common prefix of the reversed
  lists, and reverse the results again.
- The new summand replaces `alts`$_i$, while `alts`$_j$ is removed.

The Python function is a *generator*: it uses `yield` to produce the candidates one by one, so that
`factor_out` can stop as soon as it has found a good candidate.  In OCaml, we use a *lazy sequence* of type
`ast list Seq.t` instead.  Its elements are only computed when they are needed.  The helper functions `take`
and `drop` correspond to the slices `fs[:k]` and `fs[k:]`.

**Function `factor_candidates alts`**
- *Input:* `alts` is a list of summands.
- *Output:* A lazy sequence of all lists of summands that result from factoring out a common prefix or a common suffix of two summands in `alts`.

In [38]:
let factor_candidates (alts : ast list) : ast list Seq.t =
  let n = List.length alts in
  let pairs = Seq.flat_map (fun i -> Seq.map (fun j -> (i, j)) (Seq.init (n - i - 1) (fun d -> i + 1 + d)))
                           (Seq.init n Fun.id)
  in
  pairs
  |> Seq.flat_map (fun (i, j) ->
         let fi, fj = factors (List.nth alts i), factors (List.nth alts j) in
         let rest   = List.filteri (fun l _ -> l <> i && l <> j) alts in
         let by_prefix () =
           let k = common_prefix fi fj in
           if k > 0 then
             let inner = Sum (make_product (List.drop k fi), make_product (List.drop k fj)) in
             Seq.Cons (make_product (List.take k fi @ [ inner ]) :: rest, Seq.empty)
           else Seq.Nil
         in
         let by_suffix () =
           let ri, rj = List.rev fi, List.rev fj in
           let k = common_prefix ri rj in
           if k > 0 then
             let inner = Sum (make_product (List.rev (List.drop k ri)), make_product (List.rev (List.drop k rj))) in
             Seq.Cons (make_product (inner :: List.rev (List.take k ri)) :: rest, Seq.empty)
           else Seq.Nil
         in
         Seq.append by_prefix by_suffix)

val factor_candidates : ast list -> ast list Seq.t = <fun>


Factoring out does not always make a regular expression simpler: `a·b + a` has the same size as
`a·(b + ε)`.  However, the new sum `b + ε` might be simplified further.  For example, `a·b* + a` becomes
`a·(b* + ε)` and then `a·b*`.  Therefore, the function `factor_out alts` simplifies every candidate and only
accepts it if it is smaller than the original sum.  The first candidate that is smaller wins.  If there is no
such candidate, the sum is returned unchanged.

Since every candidate has fewer summands than the original sum, the recursive calls terminate.  However,
since all pairs of summands are tried, the running time grows quickly with the number of summands.  For the
small regular expressions we are interested in, this is no problem.

**Function `factor_out alts`**
- *Input:* `alts` is a list of summands.
- *Output:* An abstract syntax tree that is equivalent to the sum of the summands in `alts`: the first simplified candidate that is smaller than this sum, or the sum itself if there is no such candidate.

In [39]:
let factor_out (alts : ast list) : ast =
  let current = make_sum alts in
  let better =
    Seq.find_map (fun candidate ->
        let simplified = !simplify_ref (make_sum candidate) in
        if size simplified < size current then Some simplified else None)
      (factor_candidates alts)
  in
  Option.value better ~default:current
;;
factor_out_ref := factor_out

val factor_out : ast list -> ast = <fun>


- : unit = ()


## Putting it all Together

The function `simplify r` simplifies the abstract syntax tree `r` *bottom up*: it first simplifies the
arguments of the operator at the root of `r`, and then it simplifies the root itself.
- Atoms cannot be simplified.
- For a closure, the argument is simplified and then `simplify_star` is called.
- For a sum, we collect all summands, simplify them, and call `simplify_sum`.  If a simplified summand is a
  sum itself, it is split into its summands, so that law (b) can be applied.
- Products are handled in the same way as sums.

**Function `simplify r`**
- *Input:* `r` is an abstract syntax tree.
- *Output:* An equivalent abstract syntax tree that has been simplified bottom up.

In [40]:
let rec simplify (r : ast) : ast =
  match r with
  | Empty | Epsilon | Char _ -> r
  | Star r1 -> simplify_star (simplify r1)
  | Sum _ -> simplify_sum (List.concat_map (fun a -> summands (simplify a)) (summands r))
  | Product _ -> simplify_product (List.concat_map (fun a -> factors (simplify a)) (factors r))
;;
simplify_ref := simplify

val simplify : ast -> ast = <fun>


- : unit = ()


Simplifying a regular expression might create new opportunities for simplification.  For example, when an
inner sum shrinks to a single summand, the product around it might be simplified further.  Therefore, the
function `simplify_regexp s` calls `simplify` until the result no longer changes.  It takes a string `s`
and returns the simplified regular expression as a string.

**Function `simplify_regexp s`**
- *Input:* `s` is a string that represents a regular expression.
- *Output:* A string that represents an equivalent regular expression, which has been simplified until no law can be applied any more.

In [41]:
let simplify_regexp (s : string) : string =
  let rec loop r =
    let simplified = simplify r in
    if simplified = r then to_str r else loop simplified
  in
  loop (parse s)

val simplify_regexp : string -> string = <fun>


## Trying it Out

The function `test s` simplifies the regular expression `s` and prints both the original and the simplified
regular expression.

**Function `test s`**
- *Input:* `s` is a string that represents a regular expression.
- *Output:* `()`.  The function prints `s` together with its simplification.

In [42]:
let test (s : string) : unit =
  Printf.printf "%s ⟶  %s\n%!" (ljust s 42) (simplify_regexp s)

val test : string -> unit = <fun>


We start with the regular expressions from the section *Check your Understanding* of Chapter 2, where we
use `a` for $r$.

In [43]:
simplify_regexp "∅*"

- : string = "ε"


In [44]:
simplify_regexp "ε*"

- : string = "ε"


In [45]:
simplify_regexp "ε + a*·a"

- : string = "a*"


In [46]:
simplify_regexp "(ε + a)*"

- : string = "a*"


The following regular expressions are more complicated.  In most of them, several laws have to be applied
one after the other.  For example, in the first regular expression
- `a·∅` is simplified to `∅` by law (d), and then `ε + ∅` is simplified to `ε` by law (f),
- `b + b + ∅` is simplified to `b` by the laws (f) and (i),
- `c·ε + ∅` is simplified to `c` by the laws (e) and (f), and
- finally, the factor `ε` is dropped by law (e).

In [47]:
List.iter test
  [ "(ε + a·∅)·(b + b + ∅)*·(c·ε + ∅)";
    "a·b·c + a·b·d + a·e";
    "ε + (a + b)*·(a + b) + ∅*";
    "(a·b + a·c)·d + (a·b + a·c)·e";
    "((ε + a*)·(b·ε + ∅))* + ε";
    "x·y*·z + x·z";
    "(ε + (a·b)*·a·b)·c + ∅*·(ε + ∅)·c";
    "(((a + ∅)* + ε)*·b + a*·b)·(ε + c + ∅)";
    "(a + b·∅)*·(a + b·∅) + ((ε))";
    "((x + y)·z + (x + y)·w)* + ε·(∅ + ε)";
  ]

(ε + a·∅)·(b + b + ∅)*·(c·ε + ∅)           ⟶  b*·c
a·b·c + a·b·d + a·e                        ⟶  a·(b·(c + d) + e)
ε + (a + b)*·(a + b) + ∅*                  ⟶  (a + b)*
(a·b + a·c)·d + (a·b + a·c)·e              ⟶  a·(b + c)·(d + e)
((ε + a*)·(b·ε + ∅))* + ε                  ⟶  (a*·b)*
x·y*·z + x·z                               ⟶  x·y*·z
(ε + (a·b)*·a·b)·c + ∅*·(ε + ∅)·c          ⟶  (a·b)*·c
(((a + ∅)* + ε)*·b + a*·b)·(ε + c + ∅)     ⟶  a*·b·(ε + c)
(a + b·∅)*·(a + b·∅) + ((ε))               ⟶  a*
((x + y)·z + (x + y)·w)* + ε·(∅ + ε)       ⟶  ((x + y)·(z + w))*


- : unit = ()


Our simplifier does not find every possible simplification.  For example, the regular expressions
`ε + a·a*` and `a*` are equivalent, but none of the laws of Section 2.3 can be applied to `ε + a·a*`: law
(m) requires the closure to be the *first* factor.  In Chapter 5, we will see an algorithm that decides
whether two regular expressions are equivalent.

In [48]:
test "ε + a·a*"

ε + a·a*                                   ⟶  ε + a·a*


- : unit = ()


Finally, the parser reports syntax errors.  Instead of `try ... except`, OCaml uses `try ... with`, and the
pattern `Syntax_error msg` binds the message that the exception carries.

In [49]:
List.iter
  (fun s ->
    try ignore (parse s)
    with Syntax_error error -> Printf.printf "%s: %s\n%!" (ljust (repr s) 10) error)
  [ "a + "; "(a·b"; "a # b" ]

"a + "    : Unexpected end of input.
"(a·b"    : ')' expected, but found [].
"a # b"   : Illegal character "#" in "a # b".


- : unit = ()


## The Type Checker at Work

The function `matches r s` expects the regular expression `r` as a *string*.  The last cell of this notebook
contains a typical mistake: it calls `matches` with the *abstract syntax tree* returned by `parse`.

In the Python version of this notebook, this mistake is found by the separate type checker *basedpyright*.
*Python* itself ignores type annotations, so when the cell is executed anyway, the error message is not very
helpful: the function `parse`, which is called inside of `matches`, complains about a syntax error, since it
expects a string but gets a tuple.

In OCaml, the type checker is part of the compiler.  The last cell is rejected *before* it is executed, and
the error message points directly to the real problem: the argument `parse "a·b"` has the type `ast`, but
`matches` expects a `string`.  No part of the cell is executed.

In [50]:
matches (parse "a·b") "ab"

error: compile_error